In [1]:
# ═══════════════════════════════════════════════════════════
# NOTEBOOK 03 — MODEL EVALUATION
# Marathi Mitra — Testing the fine-tuned model
#
# This notebook:
# → Loads fine-tuned model FROM Hugging Face Hub
# → Tests on SEEN words (training set)
# → Tests on UNSEEN words (generalisation test)
# → Generates before/after comparison
# → Produces output ready for README
#
# Requires: GPU (T4 minimum)
# ═══════════════════════════════════════════════════════════

print("🔍 Marathi Mitra — Model Evaluation")
print("=" * 50)
print()
print("Key question this notebook answers:")
print("→ Did the model MEMORIZE or actually LEARN?")
print()
print("If memorized: good on seen words, fails on unseen")
print("If learned:   good on BOTH seen and unseen words")

🔍 Marathi Mitra — Model Evaluation

Key question this notebook answers:
→ Did the model MEMORIZE or actually LEARN?

If memorized: good on seen words, fails on unseen
If learned:   good on BOTH seen and unseen words


In [ ]:
# ── Install libraries ─────────────────────────────────────
!pip install -q "numpy>=2.0"
!pip install -q transformers==4.44.0
!pip install -q peft==0.12.0
!pip install -q bitsandbytes==0.45.3
!pip install -q accelerate==0.33.0
!pip install -q huggingface_hub
!pip install -q python-dotenv          # ← needed to read .env file

# ── Load credentials from .env ────────────────────────────
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv("../.env")                 # path to your .env file

HF_TOKEN    = os.getenv("HF_TOKEN")
HF_USERNAME = os.getenv("HF_USERNAME")
MODEL_REPO  = f"{HF_USERNAME}/marathi-mitra-phi3"

# Validate they loaded correctly
assert HF_TOKEN,    "❌ HF_TOKEN not found in .env"
assert HF_USERNAME, "❌ HF_USERNAME not found in .env"

login(token=HF_TOKEN)
print(f"✅ Logged in as: {HF_USERNAME}")
print(f"✅ Loading model from: {MODEL_REPO}")

In [ ]:
# ── Load directly from Hugging Face Hub ──────────────────
# This proves model works OUTSIDE of Colab training environment
# Anyone can load and use your model this way

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading fine-tuned model from Hugging Face Hub...")
print("(This is how anyone would use your model)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_REPO,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model.eval()

print(f"✅ Model loaded from hub!")
print(f"✅ Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
# ── Same generate and evaluate functions ─────────────────

def generate(word, model, tokenizer):
    prompt = f"""### Instruction:
You are Marathi Mitra, a friendly Marathi teacher for kids. \
When given an English word, teach it in Marathi with the word \
in Devanagari script, pronunciation, a simple story sentence, \
and a fun fact. Always be encouraging and kid-friendly.

### Input:
Teach me the Marathi word for: {word}

### Response:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return full_text.split("### Response:")[-1].strip()


def evaluate_output(output, expected_marathi):
    criteria = {
        "Has correct Marathi word":  expected_marathi in output,
        "Has pronunciation guide":   "How to say" in output,
        "Has example sentence":      "Example sentence" in output,
        "Has fun fact":              "Fun Fact" in output,
        "Has correct emojis":        "🌟" in output and "📢" in output,
    }
    score = sum(criteria.values()) / len(criteria) * 100
    return criteria, score


def test_word(word, expected_marathi, model, tokenizer):
    output             = generate(word, model, tokenizer)
    criteria, score    = evaluate_output(output, expected_marathi)

    print(f"\n{'─'*60}")
    print(f"📝 Word: {word.upper()}")
    print(f"\n🤖 Output:\n{output}")
    print(f"\n📊 Score: {score:.0f}%")
    for criterion, passed in criteria.items():
        print(f"   {'✅' if passed else '❌'} {criterion}")

    return output, score


print("✅ Helper functions ready")

In [ ]:
# ── Words the model trained on ────────────────────────────
# These are IN the training dataset
# Model should perform well here

SEEN_WORDS = {
    "butterfly": "फुलपाखरू",
    "mother":    "आई",
    "rain":      "पाऊस",
    "elephant":  "हत्ती",
    "school":    "शाळा",
}

print("═" * 60)
print("TEST 1 — SEEN WORDS (in training set)")
print("Expected: HIGH scores — model trained on these")
print("═" * 60)

seen_outputs = {}
seen_total   = 0

for word, expected in SEEN_WORDS.items():
    output, score      = test_word(word, expected, model, tokenizer)
    seen_outputs[word] = {"output": output, "score": score}
    seen_total        += score

seen_avg = seen_total / len(SEEN_WORDS)

print(f"\n{'═'*60}")
print(f"SEEN WORDS AVERAGE SCORE: {seen_avg:.1f}%")
print(f"{'═'*60}")

In [ ]:
# ── Words NOT in training data ────────────────────────────
# These words were NEVER shown during training
# Tests whether model GENERALISED or just memorized

# Important: these must NOT be in vocabulary.json
UNSEEN_WORDS = {
    "apple":   "सफरचंद",    # Safarchand
    "star":    "तारा",       # Taara
    "tiger":   "वाघ",        # Vaagh
    "ocean":   "समुद्र",     # Samudra
    "dance":   "नृत्य",      # Nrutya
}

print("═" * 60)
print("TEST 2 — UNSEEN WORDS (NOT in training set)")
print("Tests generalisation — did model truly learn?")
print("═" * 60)

unseen_outputs = {}
unseen_total   = 0

for word, expected in UNSEEN_WORDS.items():
    output, score         = test_word(word, expected, model, tokenizer)
    unseen_outputs[word]  = {"output": output, "score": score}
    unseen_total         += score

unseen_avg = unseen_total / len(UNSEEN_WORDS)

print(f"\n{'═'*60}")
print(f"UNSEEN WORDS AVERAGE SCORE: {unseen_avg:.1f}%")
print(f"{'═'*60}")

In [ ]:
# ── Compare seen vs unseen performance ───────────────────
print("═" * 60)
print("GENERALISATION ANALYSIS")
print("═" * 60)

print(f"\nSeen words avg:   {seen_avg:.1f}%")
print(f"Unseen words avg: {unseen_avg:.1f}%")
gap = seen_avg - unseen_avg
print(f"Gap:              {gap:.1f}%")

print()
if gap < 10:
    print("✅ EXCELLENT GENERALISATION")
    print("   Model learned the FORMAT and TASK")
    print("   not just memorizing training examples")
elif gap < 25:
    print("✅ GOOD GENERALISATION")
    print("   Model mostly generalises well")
    print("   Small drop on unseen words is normal")
elif gap < 40:
    print("⚠️  MODERATE GENERALISATION")
    print("   Model shows some memorization")
    print("   More training data would help")
else:
    print("❌ POOR GENERALISATION — OVERFITTING")
    print("   Model memorized training examples")
    print("   Does not handle unseen words well")
    print("   Solution: add more vocabulary data")

print()
print("Why gap exists:")
print("→ Seen words: model knows exact Marathi word")
print("→ Unseen words: must rely on learned patterns")
print("→ Some gap is NORMAL and EXPECTED")
print("→ Format score can still be high for unseen words")
print("  even if Marathi word is wrong")

In [ ]:
# ── Complete results table ────────────────────────────────
print("═" * 60)
print("COMPLETE EVALUATION RESULTS")
print("═" * 60)

print(f"\n{'Word':<15} {'Type':<10} {'Score':<8} {'Bar'}")
print("─" * 55)

all_scores = []

for word, data in seen_outputs.items():
    s   = data["score"]
    bar = "█" * int(s/10) + "░" * (10-int(s/10))
    print(f"{word:<15} {'SEEN':<10} {s:<8.0f} {bar}")
    all_scores.append(s)

print("─" * 55)

for word, data in unseen_outputs.items():
    s   = data["score"]
    bar = "█" * int(s/10) + "░" * (10-int(s/10))
    print(f"{word:<15} {'UNSEEN':<10} {s:<8.0f} {bar}")
    all_scores.append(s)

print("═" * 55)
overall = sum(all_scores) / len(all_scores)
overall_bar = "█" * int(overall/10) + "░" * (10-int(overall/10))
print(f"{'OVERALL':<15} {'ALL':<10} {overall:<8.1f} {overall_bar}")
print(f"\nSeen avg:   {seen_avg:.1f}%")
print(f"Unseen avg: {unseen_avg:.1f}%")
print(f"Overall:    {overall:.1f}%")

In [ ]:
# ── Generate README-ready comparison ─────────────────────
# Load base model outputs if available
# Otherwise show placeholder

print("═" * 60)
print("BEFORE vs AFTER — README CONTENT")
print("Copy this into your README.md")
print("═" * 60)

# Known baseline outputs from 02_finetune.ipynb
baseline_examples = {
    "butterfly": "सुखदाता, आपल्या प्राथ्या शब्दात... (gibberish)",
    "elephant":  'In Marathi, elephant is "गाय"... (wrong word!)',
    "school":    "विद्यालय... (wrong word, prompt leaked)",
}

print("\n```")
print("WORD: butterfly")
print()
print("❌ BEFORE (base Phi-3, no fine-tuning):")
print(baseline_examples["butterfly"])
print()
print("✅ AFTER (fine-tuned Marathi Mitra):")
print(seen_outputs["butterfly"]["output"])
print("```")

print("\n```")
print("WORD: elephant")
print()
print("❌ BEFORE:")
print(baseline_examples["elephant"])
print()
print("✅ AFTER:")
print(seen_outputs["elephant"]["output"])
print("```")

In [ ]:
# ── Summary for README and portfolio ─────────────────────
print("═" * 60)
print("EVALUATION SUMMARY")
print("═" * 60)
print()
print(f"Model:          {MODEL_REPO}")
print(f"Test words:     {len(SEEN_WORDS)} seen + {len(UNSEEN_WORDS)} unseen")
print()
print(f"Seen words:     {seen_avg:.1f}%   (words in training set)")
print(f"Unseen words:   {unseen_avg:.1f}%   (words never seen)")
print(f"Overall:        {overall:.1f}%")
print()
print("Conclusion:")
if unseen_avg >= 40:
    print("✅ Model generalises well to new Marathi vocabulary")
    print("✅ Learned the TASK not just memorized examples")
    print("✅ Ready for Phase 2 — Gradio App")
else:
    print("⚠️  Model needs more training data for better generalisation")
    print("   Recommendation: expand vocabulary.json to 100+ words")
    print("   Then retrain with best hyperparameters from Exp2/Exp3")
print()
print("Next step: Build Gradio app in app/app.py")